<a href="https://colab.research.google.com/github/Mohan102945/mohan102945.github.io/blob/main/Procurement_Optimizer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
from scipy.optimize import minimize
import numpy as np
import math

In [2]:
df_costsku = pd.read_csv('/content/df_costsku.csv')
df_demandsku = pd.read_csv('/content/df_demandsku.csv')

In [3]:
df_costsku.head()

,Unnamed: 0,SKU,COST
0,0,D1,181
1,1,D2,126
2,2,D3,144
3,3,D4,238
4,4,D5,315


In [4]:
df_costsku.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 60 entries, 0 to 59
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   Unnamed: 0  60 non-null     int64 
 1   SKU         60 non-null     object
 2   COST        60 non-null     int64 
dtypes: int64(2), object(1)
memory usage: 1.5+ KB


In [5]:
df_demandsku.head()

,Unnamed: 0,SKU,DEMAND
0,0,D1,218
1,1,D2,277
2,2,D3,62
3,3,D4,142
4,4,D5,146


In [6]:
df_demandsku.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 60 entries, 0 to 59
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   Unnamed: 0  60 non-null     int64 
 1   SKU         60 non-null     object
 2   DEMAND      60 non-null     int64 
dtypes: int64(2), object(1)
memory usage: 1.5+ KB


In [16]:
a = 42.250 # fixed transportation cost
b = -0.3975 # variable transportation cost

# Objective

In [17]:
def objective(R):
  result = 0
  for i in range(60):
    # TR cost
    result += (a + b*(df_demandsku.loc[i,'DEMAND']/R[i])) * R[i]
    #Capital Cost
    result += (df_demandsku.loc[i,'DEMAND']/(2*R[i])) * df_costsku.loc[i,'COST'] * 0.125
    #Storage cost
    result += (df_demandsku.loc[i,'DEMAND']/(2*R[i])) * 12 * 480/2000
  return result

# Constraints 1 - Maximum Inventory


In [18]:
cons = []
# Maximum Inventory
def constraint1(R):
    loop = 0
    for i in range(60):
        loop += R[i]
    result = 480 - loop
    return result
cons.append({'type':'ineq','fun':constraint1})

# Constraint 2 - Order size

In [19]:
# Add Order Size Constraints
for i in range(60):
    # Minimum Order Quantity
    c2 = lambda R : (df_demandsku.loc[i,'DEMAND']/R[i]) - 1
    cons.append({'type':'ineq','fun':c2})
    # Maximum Order Quantity
    c3 = lambda R : 400 - (df_demandsku.loc[i,'DEMAND']/R[i])
    cons.append({'type':'ineq','fun':c3})

# Initial Solution

In [20]:
R0 = [2 for i in range(60)]
print("${:,} total cost for initial guessing".format(objective(R0).round(1)))

$63,206.7 total cost for initial guessing


In [21]:
# Bound vector
b_vector = (1, 365)
bnds = tuple([b_vector for i in range(60)])

# Solution

In [22]:
sol = minimize(objective, R0, method = 'SLSQP', bounds=bnds, constraints = cons, options={'maxiter': 100})

In [23]:
print(sol)

     message: Optimization terminated successfully
     success: True
      status: 0
         fun: 28991.880313533275
           x: [ 8.112e+00  7.815e+00 ...  9.743e+00  2.789e+00]
         nit: 85
         jac: [-7.324e-04 -9.766e-04 ... -2.441e-04 -2.441e-04]
        nfev: 5286
        njev: 85
 multipliers: [ 0.000e+00  0.000e+00 ...  0.000e+00  0.000e+00]


In [24]:
print(sol.x)

[ 8.11171845  7.81477123  3.91410156  7.40501077  8.54449059  8.56038889
  3.80796984  5.03263136 10.31197909 10.87721397  8.91548568  5.53763177
  5.59634225  7.86120356  7.86690367  6.4167739   5.04128741  3.36056933
  4.43388896  6.36810587  6.46502341  3.8527386   8.3850402   1.90204485
  4.08024274  2.89446924  9.4054202   3.73509325  6.52933749  4.64031054
 11.0304098   8.3926633   5.51603807  8.16086961  4.02055225  5.89968208
  2.65010763 11.84947399  3.03291599  6.01498532 10.22880924  4.39505851
 11.11185992  6.9687984   5.09909477  1.58163311  9.78407095  3.73073315
  2.38829382  9.77742134  5.16356517  5.52052334  6.91523846  4.77173665
  9.28786853  3.17435999 11.32777274  8.24234567  9.74273163  2.78858627]
